# Day 22 — Tensors & Autograd

Autograd is how every neural net learns. It looks like magic; it's just calculus.
Let's watch PyTorch differentiate, then train a line by gradient descent.

## Tensors that remember their history

Set `requires_grad=True` and PyTorch records the operations you do, so it can replay
them backwards to get gradients. For `y = x²`, calculus says `dy/dx = 2x`:

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()          # compute dy/dx
print('x =', x.item(), ' y = x^2 =', y.item(), ' dy/dx =', x.grad.item(), '(should be 2x = 6)')

## Some noisy data from a line we'll pretend we don't know

True relationship: `y = 3x + 2` plus noise. Can gradient descent recover 3 and 2?

In [ ]:
import matplotlib.pyplot as plt
torch.manual_seed(0)

x = torch.linspace(-1, 1, 60)
y = 3.0 * x + 2.0 + 0.3 * torch.randn(60)
plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=15, alpha=0.7)
plt.title('data: y ≈ 3x + 2 + noise'); plt.xlabel('x'); plt.ylabel('y'); plt.show()

## The training loop (the heart of all of PyTorch)

**forward → loss → backward → update → zero_grad**, repeated. Watch the loss fall.

In [ ]:
w = torch.zeros((), requires_grad=True)
b = torch.zeros((), requires_grad=True)
lr, losses = 0.1, []

for epoch in range(200):
    pred = w * x + b                 # forward
    loss = ((pred - y) ** 2).mean()  # mean squared error
    loss.backward()                  # autograd fills w.grad, b.grad
    with torch.no_grad():            # update without recording it
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()   # gradients accumulate — clear them!
    losses.append(loss.item())

print(f'learned w = {w.item():.3f} (true 3.0),  b = {b.item():.3f} (true 2.0)')

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
axL.plot(losses); axL.set_title('loss curve'); axL.set_xlabel('epoch')
axL.set_ylabel('MSE'); axL.grid(alpha=0.3)

axR.scatter(x, y, s=15, alpha=0.6, label='data')
xs = torch.linspace(-1, 1, 2)
axR.plot(xs, (w * xs + b).detach(), color='crimson', lw=2, label='learned line')
axR.set_title('the fit'); axR.legend(); axR.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Takeaways

- `requires_grad` + `.backward()` gives you exact gradients for free.
- The five-line loop above is *every* PyTorch model — bigger models just have more
  parameters and fancier `forward`s.
- Forgetting `zero_grad()` is the classic beginner bug; the homework grader checks for it.

**Now go implement** `numerical_gradient` (to prove autograd is right) and `fit_line`
in `homework.py`, then run `pytest -q`.